In [1]:
import os
import pandas as pd
import polars as pl
import snapatac2 as snap
import scanpy as sc
import anndata as ad
import numpy as np

snap.__version__

'2.8.0'

In [2]:
frag_h5ad_dir = "/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess"
work_dir = '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/motor-pathway_multiome_cellbender_cluster-snrna-cr/'
script_name = 'snapatac_preprocess'
work_dir = os.path.join(work_dir, script_name)
os.makedirs(work_dir, exist_ok = True)

bw_dir = os.path.join(work_dir, 'bw')
os.makedirs(bw_dir, exist_ok = True)

In [3]:
in_dir = '/home/brad/hdd/multiome'
fragments_dict = {'ra_run2': os.path.join(in_dir, '220105_motor-pathway_run2/ra_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz'),
                 'arco_run2': os.path.join(in_dir, '220105_motor-pathway_run2/arco_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz'),
                 'ra_run3': os.path.join(in_dir, '220708_ra-arco_multiome_run3/ra_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz'),
                 'arco_run3': os.path.join(in_dir, '220708_ra-arco_multiome_run3/arco_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz'),
                 'hvc_run1': os.path.join(in_dir, '210830_hvc-nc_run1/hvc_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz'),
                 'nc_run1': os.path.join(in_dir,  '210830_hvc-nc_run1/nc_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz'),
                 'hvc_run2': os.path.join(in_dir, '220105_motor-pathway_run2/hvc_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz'),
                 'nc_run2': os.path.join(in_dir, '220105_motor-pathway_run2/nc_lonStrDom2_3p_merge_ucsc/outs/atac_fragments.tsv.gz')
                 }  

In [4]:
gtf = "/mnt/nest/assembly/lonStrDom2/ncbi/merge_gtf_with_3p/GCF_005870125.1_lonStrDom2_genomic_stringtie_FPKMthresh0.5_FPKMmaxfrac0.1_minhits5_slim_ucsc.gtf"
fasta = "/mnt/nest/assembly/lonStrDom2/ncbi/GCF_005870125.1_lonStrDom2_genomic_ucsc_only.fna"
chrom_sizes_fname = "/mnt/nest/assembly/lonStrDom2/ucsc/chrom.sizes.ucsc"
chrom_sizes = pd.read_table(chrom_sizes_fname, sep='\t', header=None)
chrom_sizes_dict = {}
for i in range(chrom_sizes.shape[0]):
    chrom_sizes_dict[chrom_sizes.at[i,0]]  = chrom_sizes.at[i,1]
lonStrDom2 = snap.genome.Genome(fasta=fasta, annotation=gtf, chrom_sizes=chrom_sizes_dict)

In [5]:

files = [os.path.join(frag_h5ad_dir, name + '.h5ad') for name in fragments_dict.keys()]
files

['/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/ra_run2.h5ad',
 '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/arco_run2.h5ad',
 '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/ra_run3.h5ad',
 '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/arco_run3.h5ad',
 '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/hvc_run1.h5ad',
 '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/nc_run1.h5ad',
 '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/hvc_run2.h5ad',
 '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/nc_run2.h5ad']

In [6]:
%%time

files = [os.path.join(frag_h5ad_dir, name + '.h5ad') for name in fragments_dict.keys()]
redo = False
if redo:
    adatas = snap.pp.import_fragments(
        [fl for fl in fragments_dict.values()],
        file=[os.path.join(frag_h5ad_dir, name + '.h5ad') for name in fragments_dict.keys()],
        chrom_sizes=lonStrDom2,
        min_num_fragments=1000,
        sorted_by_barcode=False,
    )
    snap.metrics.tsse(adatas, lonStrDom2)
    #snap.pp.filter_cells(adatas, min_tsse=7)
    snap.pp.add_tile_matrix(adatas, bin_size=5000)
    snap.pp.select_features(adatas, n_features=50000)
    #snap.pp.scrublet(adatas)
    #snap.pp.filter_doublets(adatas)
else:
    adatas = [snap.read(f) for f in files]

CPU times: user 67.3 ms, sys: 34.2 ms, total: 101 ms
Wall time: 101 ms


In [7]:
adatas[2]

AnnData object with n_obs x n_vars = 1756 x 213254 backed at '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/ra_run3.h5ad'
    obs: 'n_fragment', 'frac_dup', 'frac_mito', 'tsse'
    var: 'count', 'selected'
    uns: 'reference_sequences', 'TSS_profile', 'frac_overlap_TSS', 'library_tsse'
    obsm: 'fragment_paired'

In [8]:
%%time
data = snap.AnnDataSet(
    adatas=[(name, adata) for name, adata in zip(files, adatas)],
    filename=os.path.join(work_dir, "motor-pathway.h5ads")
)
data

CPU times: user 142 ms, sys: 32.8 ms, total: 174 ms
Wall time: 152 ms


AnnDataSet object with n_obs x n_vars = 39852 x 213254 backed at '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/motor-pathway_multiome_cellbender_cluster-snrna-cr/snapatac_preprocess/motor-pathway.h5ads'
contains 8 AnnData objects with keys: '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/ra_run2.h5ad', '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/arco_run2.h5ad', '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/ra_run3.h5ad', '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/arco_run3.h5ad', '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/hvc_run1.h5ad', '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/nc_run1.h5ad', '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/hvc_run2.h5ad', '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/snapatac_preprocess/nc_run2.h5ad'
    obs: 'sample'
    uns: 'AnnDataSet', 'reference_sequences

In [23]:
adata = data.to_adata(file=os.path.join(work_dir, "snap.h5ad"))
adata.obsm['fragment_paired'] = data.adatas.obsm['fragment_paired']

## Load Seurat data

In [25]:
adata_fname = '/ssd/brad/rstudio/multiome/motor-pathway/seurat/motor-pathway_multiome_seurat_cellbender.0.05_preprocess_cr/obj_clustered.h5ad'
adata_rna = sc.read_h5ad(adata_fname)


### Synchronize Seurat and snapatac cell names

In [26]:
samples = adata.obs['sample']
samples_df = pd.DataFrame({'samples':samples, 'cell':data.obs_names})
samples_df['sample_short'] = samples_df['samples'].str.split('/').str[-1].str.replace('.h5ad', '')
samples_df['cell_trim'] = samples_df['cell'].str.replace(r'(-[0-9])+', '', regex=True)
samples_df

/tmp/ipykernel_1146626/1928951694.py:1: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  samples = adata.obs['sample']


,samples,cell,sample_short,cell_trim
0,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACAGCCAGGCAAGC-1,ra_run2,AAACAGCCAGGCAAGC
1,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACAGCCATAGCTTG-1,ra_run2,AAACAGCCATAGCTTG
2,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACAGCCATATTGAC-1,ra_run2,AAACAGCCATATTGAC
3,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACATGCACCGGTAT-1,ra_run2,AAACATGCACCGGTAT
4,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACATGCATAAGCAA-1,ra_run2,AAACATGCATAAGCAA
...,...,...,...,...
39847,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTGTTCTTAGGGT-1,nc_run2,TTTGTGTTCTTAGGGT
39848,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTGTTCTTGTCTG-1,nc_run2,TTTGTGTTCTTGTCTG
39849,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTTGGTCACGGAT-1,nc_run2,TTTGTTGGTCACGGAT
39850,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTTGGTGTCCAAA-1,nc_run2,TTTGTTGGTGTCCAAA


In [27]:
samples_df['sample_short'].drop_duplicates()

0          ra_run2
2857     arco_run2
8190       ra_run3
9946     arco_run3
17230     hvc_run1
20399      nc_run1
26533     hvc_run2
33143      nc_run2
Name: sample_short, dtype: object

In [28]:
adata_rna.obs

,cluster,position,assignment,sample_short,run,replicate
AAACAGCCACACAATT-1,GABA-Pre,hvc,2,hvc_run1,run1,rep1
AAACCGAAGTTATGTG-1,Glut-Nido-3,hvc,2,hvc_run1,run1,rep1
AAACCGCGTTTAGCGA-1,GABA-2-1,hvc,2,hvc_run1,run1,rep1
AAACGCGCAGTAAAGC-1,Astro-2,hvc,2,hvc_run1,run1,rep1
AAACGCGCATAATGAG-1,Glut-Pre-3,hvc,2,hvc_run1,run1,rep1
...,...,...,...,...,...,...
TTTGTGTTCGCTTCTA-8,Glut-Arco-7,arco,0,arco_run3,run3,rep2
TTTGTTGGTAACGGGA-8,Glut-Arco-1,arco,2,arco_run3,run3,rep2
TTTGTTGGTAGTTGGC-8,Glut-Arco-1,arco,3,arco_run3,run3,rep2
TTTGTTGGTCATGCAA-8,Glut-Arco-1,arco,0,arco_run3,run3,rep2


In [29]:
clusters = adata_rna.obs[['sample_short', 'cluster']]
clusters['index'] = clusters.index
clusters['index_suffix'] = clusters['index'].str.extract(r'-([0-9])')
clusters['cell_trim'] = clusters['index'].str.replace(r'-[0-9]', '', regex=True)
clusters
#lusters['index_suffix'] = clusters['index_suffix'].astype(str)

#clusters_unique = clusters[['sample_short', 'index_suffix']].drop_duplicates()
#clusters_unique

/tmp/ipykernel_1146626/3226900616.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clusters['index'] = clusters.index
/tmp/ipykernel_1146626/3226900616.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clusters['index_suffix'] = clusters['index'].str.extract(r'-([0-9])')
/tmp/ipykernel_1146626/3226900616.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

,sample_short,cluster,index,index_suffix,cell_trim
AAACAGCCACACAATT-1,hvc_run1,GABA-Pre,AAACAGCCACACAATT-1,1,AAACAGCCACACAATT
AAACCGAAGTTATGTG-1,hvc_run1,Glut-Nido-3,AAACCGAAGTTATGTG-1,1,AAACCGAAGTTATGTG
AAACCGCGTTTAGCGA-1,hvc_run1,GABA-2-1,AAACCGCGTTTAGCGA-1,1,AAACCGCGTTTAGCGA
AAACGCGCAGTAAAGC-1,hvc_run1,Astro-2,AAACGCGCAGTAAAGC-1,1,AAACGCGCAGTAAAGC
AAACGCGCATAATGAG-1,hvc_run1,Glut-Pre-3,AAACGCGCATAATGAG-1,1,AAACGCGCATAATGAG
...,...,...,...,...,...
TTTGTGTTCGCTTCTA-8,arco_run3,Glut-Arco-7,TTTGTGTTCGCTTCTA-8,8,TTTGTGTTCGCTTCTA
TTTGTTGGTAACGGGA-8,arco_run3,Glut-Arco-1,TTTGTTGGTAACGGGA-8,8,TTTGTTGGTAACGGGA
TTTGTTGGTAGTTGGC-8,arco_run3,Glut-Arco-1,TTTGTTGGTAGTTGGC-8,8,TTTGTTGGTAGTTGGC
TTTGTTGGTCATGCAA-8,arco_run3,Glut-Arco-1,TTTGTTGGTCATGCAA-8,8,TTTGTTGGTCATGCAA


In [30]:
samples_dfm = pd.merge(samples_df, clusters, on = ('sample_short', 'cell_trim'), how = 'left' )
samples_dfm['index_suffix'] = samples_dfm['index_suffix'].astype(str)
samples_dfm['cell_new'] = samples_dfm[['cell_trim', 'index_suffix']].agg('-'.join, axis=1)
samples_dfmn = samples_dfm.dropna()
samples_dfmn


,samples,cell,sample_short,cell_trim,cluster,index,index_suffix,cell_new
1,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACAGCCATAGCTTG-1,ra_run2,AAACAGCCATAGCTTG,Astro-1,AAACAGCCATAGCTTG-5,5,AAACAGCCATAGCTTG-5
2,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACAGCCATATTGAC-1,ra_run2,AAACAGCCATATTGAC,Oligo,AAACAGCCATATTGAC-5,5,AAACAGCCATATTGAC-5
14,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACCGGCAGGCCATT-1,ra_run2,AAACCGGCAGGCCATT,Endo,AAACCGGCAGGCCATT-5,5,AAACCGGCAGGCCATT-5
16,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAACGCGCAATTAACC-1,ra_run2,AAACGCGCAATTAACC,Oligo,AAACGCGCAATTAACC-5,5,AAACGCGCAATTAACC-5
25,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,AAAGCCCGTTCCGGGA-1,ra_run2,AAAGCCCGTTCCGGGA,Glut-RA,AAAGCCCGTTCCGGGA-5,5,AAAGCCCGTTCCGGGA-5
...,...,...,...,...,...,...,...,...
39841,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTGGCAGCAAGAT-1,nc_run2,TTTGTGGCAGCAAGAT,Glut-Arco-2,TTTGTGGCAGCAAGAT-4,4,TTTGTGGCAGCAAGAT-4
39845,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTGTTCGTTCACC-1,nc_run2,TTTGTGTTCGTTCACC,Glut-NC-2,TTTGTGTTCGTTCACC-4,4,TTTGTGTTCGTTCACC-4
39847,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTGTTCTTAGGGT-1,nc_run2,TTTGTGTTCTTAGGGT,Glut-NC-2,TTTGTGTTCTTAGGGT-4,4,TTTGTGTTCTTAGGGT-4
39849,/hdd/jupyter/brad/snapatac/motor-pathway_multi...,TTTGTTGGTCACGGAT-1,nc_run2,TTTGTTGGTCACGGAT,Glut-NC-2,TTTGTTGGTCACGGAT-4,4,TTTGTTGGTCACGGAT-4


In [31]:
adata.obs_names = samples_dfm['cell_new']
adata.obs_names[:5]

['AAACAGCCAGGCAAGC-nan',
 'AAACAGCCATAGCTTG-5',
 'AAACAGCCATATTGAC-5',
 'AAACATGCACCGGTAT-nan',
 'AAACATGCATAAGCAA-nan']

In [32]:
import shutil

subset_path = os.path.join(work_dir, "subset.h5ad")

redo = True

if redo and os.path.exists(subset_path):
    #shutil.rmtree(subset_path)
    os.remove(subset_path)
    adata_sub = adata.subset(obs_indices=samples_dfmn['cell_new'], inplace = False)
    adata_sub.write(subset_path)
elif not os.path.exists(subset_path):
    adata_sub = adata.subset(obs_indices=samples_dfmn['cell_new'], inplace = False)
    adata_sub.write(subset_path)
else:
    adata_sub = snap.read(subset_path)

/tmp/ipykernel_1146626/1653349536.py:13: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  adata_sub = adata.subset(obs_indices=samples_dfmn['cell_new'], inplace = False)
/tmp/ipykernel_1146626/1653349536.py:13: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  adata_sub = adata.subset(obs_indices=samples_dfmn['cell_new'], inplace = False)
... storing 'sample' as categorical


In [33]:
adata_obs = adata_sub.obs_names
samples_dfmn.index = samples_dfmn['index']
samples_dfmn = samples_dfmn.loc[adata_obs]
adata_sub.obs['cluster'] = samples_dfmn['cluster']
adata_sub.obs['sample_short'] = samples_dfmn['sample_short']
adata_sub.obs['index'] = samples_dfmn['index']
adata_sub.obs['position'] = adata_sub.obs['sample_short'].str.extract('^([a-z]+)')
adata_sub.obs['position']

AAACAGCCATAGCTTG-5    ra
AAACAGCCATATTGAC-5    ra
AAACCGGCAGGCCATT-5    ra
AAACGCGCAATTAACC-5    ra
AAAGCCCGTTCCGGGA-5    ra
                      ..
TTTGTGGCAGCAAGAT-4    nc
TTTGTGTTCGTTCACC-4    nc
TTTGTGTTCTTAGGGT-4    nc
TTTGTTGGTCACGGAT-4    nc
TTTGTTGGTGTCCAAA-4    nc
Name: position, Length: 17334, dtype: object

In [37]:
adata_sub.write(subset_path)

... storing 'sample_short' as categorical
... storing 'position' as categorical
